In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU - using CPU")

False
No GPU - using CPU


In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

c:\Users\vvgk0135.DS\Desktop\mkdir medical-intent-classifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
df = pd.read_csv("../data/medquad_augmented.csv")

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["qtype"])

print(df[["qtype", "label"]].drop_duplicates().sort_values("label"))

                                        qtype  label
0                                      causes      0
3000                          exams and tests      1
6000                              information      2
9000                              inheritance      3
12000                                 outlook      4
15000                             precautions      5
18000                            side effects      6
21000                                symptoms      7
24000                               treatment      8
27000  when to contact a medical professional      9


In [9]:
import json

label_map = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
with open("../models/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

print(label_map)

{'causes': 0, 'exams and tests': 1, 'information': 2, 'inheritance': 3, 'outlook': 4, 'precautions': 5, 'side effects': 6, 'symptoms': 7, 'treatment': 8, 'when to contact a medical professional': 9}


In [7]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 21000
Validation: 4500
Test: 4500


In [8]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=10
)

print("Tokenizer and model loaded")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6681.06it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer and model loaded


In [12]:
class MedicalIntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [13]:
train_dataset = MedicalIntentDataset(train_df["question"], train_df["label"], tokenizer)
val_dataset = MedicalIntentDataset(val_df["question"], val_df["label"], tokenizer)
test_dataset = MedicalIntentDataset(test_df["question"], test_df["label"], tokenizer)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 21000
Validation dataset size: 4500
Test dataset size: 4500


In [10]:
BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Number of training batches: {len(train_loader)}")

Number of training batches: 2625


In [11]:
tiny_train = MedicalIntentDataset(
    train_df["question"].iloc[:50],
    train_df["label"].iloc[:50],
    tokenizer
)
tiny_loader = DataLoader(tiny_train, batch_size=8, shuffle=True)

batch = next(iter(tiny_loader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8])


In [12]:
df["word_count"] = df["question"].str.split().str.len()
print(df["word_count"].describe())

count    30000.000000
mean         7.939733
std          2.523988
min          2.000000
25%          6.000000
50%          8.000000
75%         10.000000
max         24.000000
Name: word_count, dtype: float64


In [13]:
MAX_LENGTH = 64

train_dataset = MedicalIntentDataset(train_df["question"], train_df["label"], tokenizer, max_length=MAX_LENGTH)
val_dataset = MedicalIntentDataset(val_df["question"], val_df["label"], tokenizer, max_length=MAX_LENGTH)
test_dataset = MedicalIntentDataset(test_df["question"], test_df["label"], tokenizer, max_length=MAX_LENGTH)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("Datasets rebuilt with max_length=64")

Datasets rebuilt with max_length=64


In [14]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
device = torch.device("cpu")
model.to(device)

print(f"Model moved to {device}")

Model moved to cpu


In [15]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    return avg_loss

In [16]:
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

In [17]:
EPOCHS = 3

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, device)
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f}")


Epoch 1/3


Evaluating: 100%|██████████| 563/563 [01:15<00:00,  7.42it/s]


Train Loss: 0.0563 | Val Loss: 0.0030 | Val Accuracy: 0.9993

Epoch 2/3


Evaluating: 100%|██████████| 563/563 [01:13<00:00,  7.62it/s]


Train Loss: 0.0135 | Val Loss: 0.1616 | Val Accuracy: 0.9284

Epoch 3/3


Evaluating: 100%|██████████| 563/563 [01:15<00:00,  7.49it/s]

Train Loss: 0.0085 | Val Loss: 0.0041 | Val Accuracy: 0.9991


In [18]:
model.save_pretrained("../models/distilbert_medical_intent")
tokenizer.save_pretrained("../models/distilbert_medical_intent")
print("Model and tokenizer saved")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.48it/s]

Model and tokenizer saved


In [19]:
test_queries = [
    "what does it mean if i have this thing in my family history",
    "is this gonna kill me or am i overreacting",
    "my chest feels tight and heavy should i worry",
    "can my kids catch this from me",
    "wat is this thing even called",
]

model.eval()
for query in test_queries:
    inputs = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=64
    )
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=1)
    confidence = probs.max().item()
    pred_label = probs.argmax().item()
    pred_class = label_encoder.inverse_transform([pred_label])[0]
    
    print(f"Query: {query}")
    print(f"Prediction: {pred_class} (confidence: {confidence:.2%})\n")

Query: what does it mean if i have this thing in my family history
Prediction: symptoms (confidence: 99.99%)

Query: is this gonna kill me or am i overreacting
Prediction: inheritance (confidence: 99.93%)

Query: my chest feels tight and heavy should i worry
Prediction: symptoms (confidence: 80.81%)

Query: can my kids catch this from me
Prediction: information (confidence: 61.50%)

Query: wat is this thing even called
Prediction: information (confidence: 99.86%)



In [14]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Done")

NameError: name 'tqdm' is not defined